[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riccardoberta/regressione/blob/main/01_il_mistero_del_valore.ipynb)

# Il mistero del valore di mercato

Siete il reparto dati di una società sportiva. Il direttore sportivo vi chiede di capire perché alcuni calciatori valgono pochi milioni e altri cifre enormi. In questo primo episodio non costruiamo ancora un modello predittivo: impariamo a guardare i dati come farebbe uno scout quantitativo.

I dati provengono da un dataset pubblico FIFA/EA Sports distribuito in formato CSV. Per rendere la lezione stabile, in questo corso non scarichiamo direttamente da Kaggle durante la lezione: usiamo una copia congelata in uno ZIP Dropbox del docente.

> **Missione** — Capire quali variabili sembrano collegate al valore di mercato di un calciatore. Non stiamo ancora facendo machine learning: stiamo costruendo intuizione sui dati.


Per cominciare, il direttore sportivo ci consegna un file enorme: i dati di tutti i calciatori del videogioco FIFA 22 (un dataset pubblico, fedele alla realtà del calcio professionistico). Apriamolo in Python e vediamo quante righe e quante colonne contiene.


In [ ]:
# Setup: imports e download del dataset (solo la prima volta)
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = (
    "https://www.dropbox.com/scl/fi/0l5n46qjwcd5moj3w7d8p/"
    "fifa.zip?rlkey=rcqhagvq5ttlvna5t5r3vn1bm&st=uzplzs5o&dl=1"
)
ZIP_PATH = Path("fifa.zip")
DATA_DIR = Path("fifa_data")

if not ZIP_PATH.exists():
    print("Scarico il dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

if not list(DATA_DIR.rglob("*.csv")):
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

# Apriamo il file principale del dataset
csv_files = (
    list(DATA_DIR.rglob("players_22.csv"))
    or list(DATA_DIR.rglob("*players*.csv"))
)
raw = pd.read_csv(csv_files[0], low_memory=False)
print(f"Caricato: {raw.shape[0]} giocatori, {raw.shape[1]} colonne")
raw.head()


Quel file ha decine di colonne: troppe per essere utili in una prima ricognizione. Riduciamolo all'essenziale per la nostra missione: identità del giocatore, età, qualità tecniche (overall, potential), salario, valore di mercato, club e ruolo. Togliamo anche le righe in cui manca il valore di mercato (non possiamo studiare quello che non sappiamo) e l'1% di giocatori più costosi (i Messi e i Ronaldo del dataset), che altrimenti distorcerebbero i grafici.


In [ ]:
# Creiamo una versione didattica piccola e pulita
wanted_columns = [
    "short_name", "age", "overall", "potential", "wage_eur", "value_eur",
    "club_name", "league_name", "player_positions"
]

available = [c for c in wanted_columns if c in raw.columns]
df = raw[available].copy()

# Manteniamo solo i giocatori con valore noto e positivo.
df = df.dropna(subset=["value_eur", "age", "overall", "potential"])
df = df[df["value_eur"] > 0]

# Riduciamo l'effetto dei super-outlier per i primi grafici didattici.
df = df[df["value_eur"] <= df["value_eur"].quantile(0.99)]

# Per leggibilita' useremo spesso il valore in milioni di euro.
df["value_mln_eur"] = df["value_eur"] / 1_000_000
if "wage_eur" in df.columns:
    df["wage_k_eur"] = df["wage_eur"] / 1_000

print("Forma del dataset didattico:", df.shape)
df.head(10)

> **Dataset** — Ogni riga rappresenta un calciatore. Le colonne sono caratteristiche osservabili: eta', overall, potential, salario, club e valore economico. In un problema di regressione il valore economico sara' la variabile da prevedere, mentre le altre colonne saranno gli indizi.


### Teoria — Come scriviamo il problema in matematica

Abbiamo $n$ giocatori. Per ognuno raccogliamo $p$ caratteristiche numeriche (età, overall, potential, ...) e un valore di mercato.

- Le caratteristiche del giocatore $i$ formano un vettore $\mathbf{x}_i = (x_{i,1}, x_{i,2}, \dots, x_{i,p})$.
- Il suo valore di mercato è un numero $y_i \in \mathbb{R}$.
- Tutti i dati insieme: la **matrice degli input** $X \in \mathbb{R}^{n \times p}$ e il **vettore dei target** $\mathbf{y} \in \mathbb{R}^n$.

Un problema di **regressione** consiste nel trovare una funzione $f$ tale che $f(\mathbf{x}_i) \approx y_i$ per tutti i giocatori. Il termine $\approx$ è importante: non vogliamo (e non possiamo) prevedere il valore esatto, vogliamo prevederlo *bene in media*.


## Primo sguardo: chi sono i giocatori più costosi?

Prima ancora di costruire grafici e modelli, mettiamoci nei panni di un osservatore: chi sta in cima alla classifica per valore di mercato? Vedere chi sono i 15 giocatori più costosi ci dà già qualche idea di cosa renda un calciatore *prezioso* agli occhi del mercato.


In [ ]:
df.sort_values("value_eur", ascending=False)[[
    "short_name", "age", "overall", "potential",
    "value_mln_eur", "club_name", "player_positions"
]].head(15)

---

> **Domanda per la classe** — Guardando questa tabella, quale variabile vi sembra piu' importante: eta', overall o potential? Scrivete un'ipotesi prima di guardare i grafici.


## Età e valore

L'età è la prima caratteristica che ogni scout guarda. Disegniamo un punto per ogni giocatore (un puntino = un calciatore) con l'età sull'asse orizzontale e il valore di mercato sull'asse verticale. Se l'età avesse un legame forte col valore, il grafico mostrerebbe una struttura riconoscibile.


In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["age"], df["value_mln_eur"], alpha=0.25)
plt.xlabel("Eta'")
plt.ylabel("Valore di mercato [milioni di euro]")
plt.title("Eta' vs valore di mercato")
plt.grid(True, alpha=0.3)
plt.show()

> **Relazione tra variabili** — Quando disegniamo un grafico con una variabile sull'asse x e una sull'asse y, stiamo chiedendo se conoscere x ci aiuta a prevedere y. Se i punti formano una struttura riconoscibile, allora x contiene informazione utile su y. Se i punti sono completamente sparsi, x da sola probabilmente non basta.


## Overall e valore

Ora la qualità tecnica complessiva, l'**overall**: è il numero (da 0 a 99) con cui FIFA riassume il livello attuale del giocatore. Aspettiamoci una nuvola più nitida rispetto a quella dell'età.


In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["overall"], df["value_mln_eur"], alpha=0.25)
plt.xlabel("Overall rating")
plt.ylabel("Valore di mercato [milioni di euro]")
plt.title("Overall vs valore di mercato")
plt.grid(True, alpha=0.3)
plt.show()

## Potential e valore

Infine il **potential**: il livello che ogni giocatore *potrebbe* raggiungere in futuro secondo i talent scout di EA Sports. È un numero più speculativo dell'overall, ma il mercato lo guarda con attenzione perché i club comprano talento futuro, non solo presente.


In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["potential"], df["value_mln_eur"], alpha=0.25)
plt.xlabel("Potential rating")
plt.ylabel("Valore di mercato [milioni di euro]")
plt.title("Potential vs valore di mercato")
plt.grid(True, alpha=0.3)
plt.show()

> **Correlazione intuitiva** — Se al crescere di una variabile tende a crescere anche il valore, diciamo che c'e' una correlazione positiva. Attenzione: correlazione non significa causalita'. Qui non stiamo dicendo che aumentare artificialmente l'overall causa automaticamente un prezzo piu' alto; stiamo osservando che nel dataset i due numeri si muovono spesso insieme.


## Una misura numerica dell'associazione

### Teoria — Coefficiente di correlazione

Per misurare quanto due variabili numeriche si muovono insieme usiamo il **coefficiente di correlazione di Pearson**:

$\displaystyle r_{x,y} \;=\; \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n} (x_i - \bar{x})^2}\;\sqrt{\sum_{i=1}^{n} (y_i - \bar{y})^2}}$

dove $\bar{x}$ e $\bar{y}$ sono le medie. Il valore $r$ sta sempre nell'intervallo $[-1, +1]$:

- $r \approx +1$: al crescere di $x$ tende a crescere anche $y$ (correlazione positiva).
- $r \approx -1$: al crescere di $x$, $y$ tende a calare (correlazione negativa).
- $r \approx 0$: nessuna relazione *lineare* evidente (potrebbe esserci una relazione non lineare).

**Attenzione**: una correlazione alta non implica una relazione di causa-effetto.


I tre grafici ci hanno dato un'intuizione, ma vogliamo un numero. Applichiamo la formula di Pearson al nostro dataset: per ogni variabile numerica calcoliamo la correlazione con il valore di mercato e ordiniamo dalla più *amica* del valore alla più *indifferente*.


In [ ]:
numeric_cols = [
    c for c in ["age", "overall", "potential", "wage_eur", "value_mln_eur"]
    if c in df.columns
]
corr = (
    df[numeric_cols].corr(numeric_only=True)["value_mln_eur"]
    .sort_values(ascending=False)
)
corr

---

> **Cosa dovremmo aver capito** — Il valore di mercato non dipende da una sola variabile. Overall e potential sono molto informativi, l'eta' ha un effetto piu' sottile, e il salario e' spesso collegato al valore. Nel prossimo notebook costruiremo il primo modello: una formula che prova a prevedere il valore.
